In [11]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/pgh311.db")
df = pd.read_sql("SELECT * FROM model_data", con)
df.shape

(46875, 4)

In [12]:
# 1. Overall 30-day closure rate (should be ~0.727)
df["closed_within_30d"].mean()

np.float64(0.726976)

In [13]:
# 2. Quick structure check: 88 neighborhoods, 8 subjects, only 0s and 1s
print(df["neighborhood"].nunique(), "neighborhoods")
print(df["subject"].nunique(), "subjects")
print(df["closed_within_30d"].value_counts())

88 neighborhoods
8 subjects
closed_within_30d
1    34077
0    12798
Name: count, dtype: int64


In [14]:
# 3. Closure rate and count by subject, slowest first
by_subject = (
    df.groupby("subject")["closed_within_30d"]
      .agg(rate="mean", n="size")
      .sort_values("rate")
)
by_subject["rate"] = (by_subject["rate"] * 100).round(1)
by_subject

,rate,n
subject,,
City Tree Maintenance,31.6,3202
Overgrowth,35.5,3040
Street Light Concerns,55.1,4894
Potholes,60.2,9121
Litter on Public Property,60.8,3473
Street Cleaning/Sweeping,62.0,1955
Repair or Replace a Sign,87.4,3115
Missed Pick Up,98.2,18075


In [15]:
# 4. Gap between the fastest and slowest service, in percentage points
by_subject["rate"].max() - by_subject["rate"].min()

np.float64(66.6)

In [16]:
df["neighborhood"].value_counts().head()

neighborhood
Squirrel Hill South    1952
Brookline              1950
Carrick                1821
Bloomfield             1609
Shadyside              1578
Name: count, dtype: int64

In [17]:
import numpy as np
import statsmodels.formula.api as smf

model = smf.logit(
    "closed_within_30d ~ C(neighborhood, Treatment(reference='Squirrel Hill South')) + C(subject)",
    data=df
).fit()

Optimization terminated successfully.
         Current function value: 0.408384
         Iterations 8


In [18]:
odds_ratios = np.exp(model.params)
odds_ratios[odds_ratios.index.str.contains("subject")]

C(subject)[T.Litter on Public Property]      3.091444
C(subject)[T.Missed Pick Up]               127.306980
C(subject)[T.Overgrowth]                     1.041822
C(subject)[T.Potholes]                       3.782565
C(subject)[T.Repair or Replace a Sign]      16.181019
C(subject)[T.Street Cleaning/Sweeping]       4.033342
C(subject)[T.Street Light Concerns]          2.699196
dtype: float64

In [19]:
# Neighborhood odds ratios with 95% confidence intervals
is_nbhd = model.params.index.str.contains("neighborhood")
ci = model.conf_int()                       # columns 0 (lower) and 1 (upper), on the log-odds scale

nbhd_or = pd.DataFrame({
    "odds_ratio": np.exp(model.params[is_nbhd]),
    "lower":      np.exp(ci.loc[is_nbhd, 0]),
    "upper":      np.exp(ci.loc[is_nbhd, 1]),
})

# Tidy the row names: "C(neighborhood, ...)[T.Bloomfield]" -> "Bloomfield"
nbhd_or.index = nbhd_or.index.str.extract(r"\[T\.(.+)\]", expand=False)

nbhd_or = nbhd_or.sort_values("odds_ratio")

# How many are clearly different from Squirrel Hill South?
below = (nbhd_or["upper"] < 1).sum()        # whole interval below 1 -> clearly slower
above = (nbhd_or["lower"] > 1).sum()        # whole interval above 1 -> clearly faster
print(f"Clearly slower: {below}, clearly faster: {above}, out of {len(nbhd_or)}")

nbhd_or.head(5).round(2)

Clearly slower: 7, clearly faster: 51, out of 87


,odds_ratio,lower,upper
Homewood West,0.55,0.37,0.82
Hays,0.68,0.44,1.04
Homewood North,0.68,0.51,0.91
East Hills,0.70,0.52,0.94
Polish Hill,0.74,0.55,0.99
